# Tune HMM_MCMC, HMM_MCMC_BW, HMM_MCMC_RAM, HMM_MCMC_GP_MALA, HMM_MCMC_TEST, and HMM_MCMC_DREAM for yahpo-gym lcbench

Meta-optimization of HMM hyperparameters on the **lcbench** surrogate benchmark.

**Method:** Optuna TPE meta-search over HMM hyperparameters, then validation on held-out seeds.

**Meta-objective:** maximize mean best `val_accuracy` over multiple lcbench instances × tuning seeds.

**Algorithms:** `HMM_MCMC` (base), `HMM_MCMC_BW` (Baum-Welch, no spline), `HMM_MCMC_RAM` (BW + diagonal RAM on KDE mixture), `HMM_MCMC_GP_MALA` (BW + GP + MALA + RAM), `HMM_MCMC_TEST` (Baum-Welch + spline), and `HMM_MCMC_DREAM` (DREAM + soft HMM + Baum-Welch).

**Tuning modes:** `SMOKE_MODE` (quick check) · `THOROUGH_MODE` (120+50 meta-trials, wider search, warm-start + refine)


In [1]:
# =============================================================================
# Configuration
# =============================================================================
from __future__ import annotations

from pathlib import Path
import os

if Path.cwd().name == "experiments":
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

SMOKE_MODE = False  # set False for full tuning run
THOROUGH_MODE = True  # wider search, more trials/seeds, warm-start + refine phase

LCBENCH_INSTANCES = ["3945"] if SMOKE_MODE else ["3945", "7593", "126026"]
if SMOKE_MODE:
    TUNE_SEEDS = [42]
    VAL_SEEDS = [44]
    N_EVALS = 60
    N_META_TRIALS = 20
    N_REFINE_TRIALS = 0
    META_STARTUP_TRIALS = 8
elif THOROUGH_MODE:
    TUNE_SEEDS = [42, 43, 44]
    VAL_SEEDS = [45, 46, 47, 48]
    N_EVALS = 200
    N_META_TRIALS = 120
    N_REFINE_TRIALS = 50
    META_STARTUP_TRIALS = 30
else:
    TUNE_SEEDS = [42, 43]
    VAL_SEEDS = [44, 45, 46]
    N_EVALS = 150
    N_META_TRIALS = 60
    N_REFINE_TRIALS = 0
    META_STARTUP_TRIALS = 15

META_SEED = 42
WARM_START_FROM_BEST = True  # enqueue previous best_params_hmm_mcmc_dream.json
META_BLEND_BEST = True  # blend mean + max seed score in meta-objective
META_BLEND_WEIGHT = 0.3  # weight on max(best seed) vs mean across seeds

DATA_PATH = Path(os.environ.get("YAHPO_DATA_PATH", REPO_ROOT / "yahpo_data"))
PLOTS_DIR = REPO_ROOT / "experiments" / "yahpo_tuning_plots"
RESULTS_DIR = REPO_ROOT / "experiments" / "yahpo_tuning_results"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

USE_TQDM = True

LCBENCH_CFG = {
    "target": "val_accuracy",
    "fidelity": "epoch",
}

HMM_FIXED = dict(
    n_chains=1,
    orchestrate_every=1000,
    T_min=None,
    hmm_obs_epsilon=1e-8,
    hmm_lambda_noise=0.01,
    rejection_streak=10,
)

HMM_BW_FIXED = dict(
    use_baum_welch=True,
    bw_n_em_iters=3,
    bw_max_len=64,
    show_progress=False,
    verbose_history=False,
)

HMM_TEST_FIXED = dict(
    use_baum_welch=True,
    use_spline_proposal=True,
    bw_n_em_iters=3,
    bw_max_len=64,
    show_progress=False,
    verbose_history=False,
)

HMM_DEFAULT = dict(
    n_init=32,
    T_mcmc=0.01,
    sigma_fraction=0.0055,
    wide_sigma_fraction=0.5,
    temperature=0.3,
    hmm_window=4,
    clone_noise=0.05,
    burnin_fraction=0.0,
    p_cat_step=0.30,
    kde_tau=0.05,
    anneal_T=True,
    **HMM_FIXED,
)

HMM_BW_DEFAULT = dict(
    **HMM_DEFAULT,
    use_baum_welch=True,
    bw_refit_every=5,
    bw_min_obs=12,
    bw_prior_strength=25.0,
    bw_exploit_prior_scale=3.0,
    show_progress=False,
    verbose_history=False,
)

HMM_TEST_DEFAULT = dict(
    **HMM_DEFAULT,
    use_baum_welch=True,
    bw_refit_every=5,
    bw_min_obs=12,
    bw_prior_strength=25.0,
    bw_exploit_prior_scale=3.0,
    use_spline_proposal=True,
    spline_knots=60,
    spline_floor=0.05,
    spline_min_archive=50,
    locality_sigma_fraction=0.08,
    spline_mix_scale=1.0,
    show_progress=False,
    verbose_history=False,
)

HMM_RAM_FIXED = dict(
    **HMM_BW_FIXED,
    use_ram=True,
    # Fixed to reduce meta-tuning overfit (see validation vs default analysis)
    n_init=32,
    wide_sigma_fraction=0.5,
    burnin_fraction=0.0,
    bw_min_obs=12,
    bw_prior_strength=25.0,
    bw_refit_every=5,
    bw_exploit_prior_scale=3.0,
    ram_target_accept=0.234,
    ram_precond_init="archive",
)

HMM_RAM_DEFAULT = dict(
    **HMM_BW_DEFAULT,
    use_ram=True,
    ram_target_accept=0.234,
    ram_gamma=0.7,
    ram_s_min=0.005,
    ram_s_max=0.5,
    ram_precond_init="archive",
)

HMM_GP_MALA_FIXED = dict(
    **HMM_BW_FIXED,
    use_gp_mala=True,
    n_init=12,
    wide_sigma_fraction=0.5,
    burnin_fraction=0.0,
    bw_min_obs=12,
    bw_prior_strength=25.0,
    bw_refit_every=5,
    bw_exploit_prior_scale=3.0,
    ram_target_accept=0.574,
    ram_precond_init="archive",
    ram_s_min=0.02,
    ram_s_max=0.5,
    gp_min_obs=12,
    gp_max_obs=256,
    gp_matern_nu=2.5,
    gp_explore_mode="thompson",
    gp_target_temperature=1.0,
    gp_refit_every=1,
    gp_argmax_every=1,
    gp_argmax_restarts=5,
    gp_argmax_steps=50,
    tr_enable=True,
    tr_length_init=0.8,
    tr_length_min=0.05,
    tr_length_max=1.6,
    tr_success_tol=3,
    tr_failure_tol=4,
    tr_improve_tol=1e-3,
    acq_anneal=True,
    acq_greedy_after=0.85,
    tr_restart_candidates=64,
    tr_restart_kappa_start=3.0,
    tr_restart_kappa_end=0.5,
    bo_warmup_frac=0.25,
)

HMM_GP_MALA_DEFAULT = dict(
    **HMM_BW_DEFAULT,
    use_gp_mala=True,
    n_init=12,
    ram_target_accept=0.574,
    ram_gamma=0.7,
    ram_s_min=0.02,
    ram_s_max=0.5,
    ram_precond_init="archive",
    gp_refit_every=1,
    gp_min_obs=12,
    gp_max_obs=256,
    gp_matern_nu=2.5,
    gp_kappa_exploit=1.0,
    gp_kappa_explore=2.0,
    gp_explore_mode="thompson",
    gp_target_temperature=1.0,
    gp_argmax_every=1,
    gp_argmax_restarts=5,
    gp_argmax_steps=50,
    mala_step_size=0.1,
    tr_enable=True,
    tr_length_init=0.8,
    tr_length_min=0.05,
    tr_length_max=1.6,
    tr_success_tol=3,
    tr_failure_tol=4,
    tr_improve_tol=1e-3,
    acq_anneal=True,
    acq_greedy_after=0.85,
    tr_restart_candidates=64,
    tr_restart_kappa_start=3.0,
    tr_restart_kappa_end=0.5,
    bo_warmup_frac=0.25,
)

HMM_HMC_FIXED = dict(
    show_progress=False,
    verbose_history=False,
    n_init=16,
    gp_min_obs=12,
    gp_max_obs=256,
    gp_refit_every=1,
    gp_noise_floor=1e-6,
    bo_warmup_frac=0.25,
    exploit_argmax_every=1,
    gp_argmax_restarts=20,
    gp_argmax_raw_samples=256,
    acq_greedy_after=0.97,
    explore_beta=4.0,
    gp_subsample=False,
    burst_steps_escape=64,
    burst_steps_stuck=32,
    hmm_switch_confirm=2,
    p_cat_step=0.0,
)

HMM_HMC_DEFAULT = dict(
    n_init=16,
    rejection_streak=10,
    hmm_window=8,
    hmm_refit_every=5,
    hmm_min_obs=12,
    hmm_n_em_iters=3,
    hmm_prior_strength=25.0,
    gp_min_obs=12,
    gp_max_obs=256,
    gp_refit_every=1,
    gp_noise_floor=1e-6,
    kappa_exploit=1.0,
    kappa_explore=3.0,
    kappa_escape=5.0,
    T_exploit=0.3,
    T_explore=1.0,
    T_escape=2.0,
    step_size_init=0.1,
    target_accept=0.8,
    nuts_max_tree_depth=10,
    bo_warmup_frac=0.25,
    exploit_argmax_every=1,
    gp_argmax_restarts=20,
    gp_argmax_raw_samples=256,
    acq_greedy_after=0.97,
    explore_beta=4.0,
    gp_subsample=False,
    burst_steps_escape=64,
    burst_steps_stuck=32,
    hmm_switch_confirm=2,
    p_cat_step=0.0,
)

# Which algorithms to meta-tune (default: HMC; add other variants to re-tune)
TUNE_ALGORITHMS = ["HMM_MCMC_HMC"]

HMM_DREAM_FIXED = dict(
    show_progress=False,
    verbose_history=False,
)

HMM_DREAM_DEFAULT = {
    **HMM_DEFAULT,
    "n_chains": 4,
    "orchestrate_every": 5,
    "orchestrate_patience": 15,
    "p_dream": 0.5,
    "dream_n_pairs": 1,
    "dream_cr": 0.9,
    "dream_gamma1_prob": 0.1,
    "dream_eps": 1e-3,
    "dream_min_pop": 4,
    "dream_diversity_frac": 0.0,
    "bw_refit_every": 5,
    "bw_min_obs": 12,
    "bw_prior_strength": 25.0,
    "bw_exploit_prior_scale": 3.0,
    "show_progress": False,
    "verbose_history": False,
}

_inner_runs = N_META_TRIALS * len(LCBENCH_INSTANCES) * len(TUNE_SEEDS)
_refine_runs = N_REFINE_TRIALS * len(LCBENCH_INSTANCES) * len(TUNE_SEEDS)
print(f"REPO_ROOT={REPO_ROOT}")
print(f"SMOKE_MODE={SMOKE_MODE}, THOROUGH_MODE={THOROUGH_MODE}")
print(f"Instances={LCBENCH_INSTANCES}, tune_seeds={TUNE_SEEDS}, val_seeds={VAL_SEEDS}")
print(f"N_EVALS={N_EVALS}, N_META_TRIALS={N_META_TRIALS}, N_REFINE_TRIALS={N_REFINE_TRIALS}")
print(f"Estimated inner HMM runs: coarse={_inner_runs}, refine={_refine_runs}")
print(f"DATA_PATH={DATA_PATH}")


REPO_ROOT=c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL
SMOKE_MODE=False, THOROUGH_MODE=True
Instances=['3945', '7593', '126026'], tune_seeds=[42, 43, 44], val_seeds=[45, 46, 47, 48]
N_EVALS=200, N_META_TRIALS=120, N_REFINE_TRIALS=50
Estimated inner HMM runs: coarse=1080, refine=450
DATA_PATH=c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\yahpo_data


In [2]:
# =============================================================================
# Dependencies + yahpo data path + ConfigSpace monkeypatch
# =============================================================================
import importlib
import subprocess
import sys

REQUIRED = [
    ("optuna", "optuna>=3.6"),
    ("optuna_integration", "optuna-integration"),
]


def _is_importable(module_name: str) -> bool:
    try:
        importlib.import_module(module_name)
        return True
    except ImportError:
        return False


missing = [spec for mod, spec in REQUIRED if not _is_importable(mod)]
if missing:
    print("Installing:", " ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing, "-q"])

from yahpo_gym import local_config

local_config.init_config() if not local_config.settings_path.exists() else None
local_config.set_data_path(str(DATA_PATH))

encoding_probe = DATA_PATH / "lcbench" / "encoding.json"
if not encoding_probe.is_file():
    raise FileNotFoundError(
        f"yahpo data not found at {DATA_PATH}. "
        "Clone https://github.com/slds-lmu/yahpo_data.git"
    )
print("yahpo data OK:", encoding_probe)


yahpo data OK: c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\yahpo_data\lcbench\encoding.json


In [3]:
# =============================================================================
# Imports + repo bootstrap
# =============================================================================
import json
import sys
from dataclasses import dataclass
from typing import Any, Type

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import ConfigSpace as CS
import ConfigSpace.hyperparameters as CSH
import optuna
from optuna.samplers import TPESampler
from yahpo_gym import benchmark_set

if not hasattr(CS.ConfigurationSpace, "_sort_hyperparameters"):
    CS.ConfigurationSpace._sort_hyperparameters = lambda self: None

_root = str(REPO_ROOT)
if _root not in sys.path:
    sys.path.insert(0, _root)

from hpo_rl.baselines.HMM_MCMC import HMM_MCMC
from hpo_rl.baselines.HMM_MCMC_TEST import HMM_MCMC_TEST
from hpo_rl.baselines.HMM_MCMC_BW import HMM_MCMC_BW
from hpo_rl.baselines.HMM_MCMC_DREAM import HMM_MCMC_DREAM
from hpo_rl.baselines.HMM_MCMC_RAM import HMM_MCMC_RAM
from hpo_rl.baselines.HMM_MCMC_GP_MALA import HMM_MCMC_GP_MALA
from hpo_rl.baselines.HMM_MCMC_HMC import HMM_MCMC_HMC

optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})


In [4]:
# =============================================================================
# yahpo lcbench adapter helpers
# =============================================================================


@dataclass
class ScenarioContext:
    instance: str
    bench: Any
    target: str
    fidelity: str
    max_fidelity: float
    fidelity_is_int: bool
    hpo_rl_space: dict


def _hp_bounds(hp: CSH.Hyperparameter) -> tuple[float, float]:
    return float(hp.lower), float(hp.upper)


def _hp_max_constant(hp: CSH.Hyperparameter):
    if isinstance(hp, CSH.UniformIntegerHyperparameter):
        return int(hp.upper)
    return float(hp.upper)


def configspace_to_hpo_rl_space(cs: CS.ConfigurationSpace, exclude: set[str] | None = None) -> dict:
    exclude = exclude or set()
    space: dict = {}
    for hp in cs.get_hyperparameters():
        name = hp.name
        if name in exclude or isinstance(hp, CSH.Constant):
            continue
        if isinstance(hp, CSH.UniformFloatHyperparameter):
            space[name] = {
                "type": "float",
                "values": [float(hp.lower), float(hp.upper)],
                "log": bool(hp.log),
            }
        elif isinstance(hp, CSH.UniformIntegerHyperparameter):
            space[name] = {
                "type": "int",
                "values": [int(hp.lower), int(hp.upper)],
                "log": bool(hp.log),
            }
        elif isinstance(hp, CSH.CategoricalHyperparameter):
            space[name] = {"type": "categorical", "values": list(hp.choices)}
        elif isinstance(hp, CSH.OrdinalHyperparameter):
            space[name] = {"type": "categorical", "values": list(hp.sequence)}
        else:
            raise TypeError(f"Unsupported hyperparameter: {name} ({type(hp)})")
    return space


def build_lcbench_context(instance: str) -> ScenarioContext:
    bench = benchmark_set.BenchmarkSet("lcbench")
    bench.set_instance(instance)
    bench.check = False

    target = LCBENCH_CFG["target"]
    if target not in bench.targets:
        raise ValueError(f"Target {target} not in {bench.targets}")

    fidelity = LCBENCH_CFG["fidelity"]
    fspace = bench.get_fidelity_space()
    f_hp = fspace[fidelity]
    _, max_f = _hp_bounds(f_hp)
    fidelity_is_int = isinstance(f_hp, CSH.UniformIntegerHyperparameter)

    for fp in bench.config.fidelity_params:
        if fp == fidelity:
            continue
        bench.set_constant(fp, _hp_max_constant(fspace[fp]))

    opt_space = bench.get_opt_space(drop_fidelity_params=True)
    hpo_rl_space = configspace_to_hpo_rl_space(opt_space, exclude=set(bench.config.fidelity_params))

    return ScenarioContext(
        instance=instance,
        bench=bench,
        target=target,
        fidelity=fidelity,
        max_fidelity=max_f,
        fidelity_is_int=fidelity_is_int,
        hpo_rl_space=hpo_rl_space,
    )


def evaluate_at_max_fidelity(ctx: ScenarioContext, config: dict) -> float:
    cfg = dict(config)
    val = int(round(ctx.max_fidelity)) if ctx.fidelity_is_int else float(ctx.max_fidelity)
    cfg[ctx.fidelity] = val
    return float(ctx.bench.objective_function(cfg)[0][ctx.target])


contexts: dict[str, ScenarioContext] = {}
for inst in tqdm(LCBENCH_INSTANCES, desc="Building lcbench contexts", disable=not USE_TQDM):
    ctx = build_lcbench_context(inst)
    contexts[inst] = ctx
    print(
        f"  instance={inst}, target={ctx.target}, max_{ctx.fidelity}={ctx.max_fidelity}, "
        f"n_hp={len(ctx.hpo_rl_space)}"
    )


Building lcbench contexts:   0%|          | 0/3 [00:00<?, ?it/s]

c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\.venv\Lib\site-packages\ConfigSpace\configuration_space.py:1038: UserWarning: The field 'default' should be 'default_value' !
Found in item {'name': 'OpenML_task_id', 'choices': ['3945', '7593', '34539', '126025', '126026', '126029', '146212', '167104', '167149', '167152', '167161', '167168', '167181', '167184', '167185', '167190', '167200', '167201', '168329', '168330', '168331', '168335', '168868', '168908', '168910', '189354', '189862', '189865', '189866', '189873', '189905', '189906', '189908', '189909'], 'probabilities': None}
  return decoder(item, cs, _dec)
c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\.venv\Lib\site-packages\ConfigSpace\configuration_space.py:1038: UserWarning: The field 'default' should be 'default_value' !
Found in item {'name': 'batch_size', 'log': True, 'lower': 16, 'upper': 512}
  return decoder(item, cs, _dec)
c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\.venv\Lib\site-pack

  instance=3945, target=val_accuracy, max_epoch=52.0, n_hp=7
  instance=7593, target=val_accuracy, max_epoch=52.0, n_hp=7
  instance=126026, target=val_accuracy, max_epoch=52.0, n_hp=7


In [5]:
# =============================================================================
# HMM runners + meta-objective
# =============================================================================


def run_hmm_once(
    algo_cls: Type,
    ctx: ScenarioContext,
    hmm_params: dict,
    n_evals: int,
    seed: int,
) -> float:
    """Run one HMM optimization; return best val_accuracy."""
    best_acc = -np.inf

    def objective(config_dict: dict) -> float:
        nonlocal best_acc
        acc = evaluate_at_max_fidelity(ctx, config_dict)
        best_acc = max(best_acc, acc)
        return -acc

    np.random.seed(seed)
    params = dict(hmm_params)
    if algo_cls in (HMM_MCMC_BW, HMM_MCMC_RAM, HMM_MCMC_GP_MALA, HMM_MCMC_HMC, HMM_MCMC_TEST, HMM_MCMC_DREAM):
        params.setdefault("show_progress", False)
        params.setdefault("verbose_history", False)
    else:
        params.pop("show_progress", None)
        params.pop("verbose_history", None)

    alg = algo_cls(
        objective_func=objective,
        budget=n_evals,
        dict_to_optimize=ctx.hpo_rl_space,
        seed=seed,
        **params,
    )
    alg.main_loop()
    return float(best_acc)


def meta_objective(
    algo_cls: Type,
    hmm_params: dict,
    seeds: list[int],
    desc: str = "",
) -> float:
    """Mean (or blended) best val_accuracy over instances x seeds."""
    scores: list[float] = []
    runs = [(inst, seed) for inst in LCBENCH_INSTANCES for seed in seeds]
    pbar = tqdm(runs, desc=desc, leave=False, disable=not USE_TQDM)
    for inst, seed in pbar:
        acc = run_hmm_once(algo_cls, contexts[inst], hmm_params, N_EVALS, seed)
        scores.append(acc)
        pbar.set_postfix(inst=inst, seed=seed, acc=f"{acc:.4f}")
    mean_score = float(np.mean(scores))
    if META_BLEND_BEST and scores:
        best_score = float(np.max(scores))
        w = float(META_BLEND_WEIGHT)
        return (1.0 - w) * mean_score + w * best_score
    return mean_score


In [6]:
# =============================================================================
# Optuna meta-search spaces
# =============================================================================


def _clip(v: float, lo: float, hi: float) -> float:
    return float(max(lo, min(hi, v)))


def _narrow_int(trial: optuna.Trial, name: str, center: int, lo: int, hi: int, frac: float = 0.35) -> int:
    span = max(1, int(round(abs(center) * frac)))
    return trial.suggest_int(name, max(lo, center - span), min(hi, center + span))


def _narrow_float(
    trial: optuna.Trial,
    name: str,
    center: float,
    lo: float,
    hi: float,
    *,
    log: bool = False,
    frac: float = 0.45,
) -> float:
    if log:
        c = max(center, lo * 1.01)
        lo_n = _clip(c * (1.0 - frac), lo, hi)
        hi_n = _clip(c * (1.0 + frac), lo, hi)
        if lo_n >= hi_n:
            lo_n, hi_n = lo, hi
        return trial.suggest_float(name, lo_n, hi_n, log=True)
    span = max((hi - lo) * 0.05, abs(center) * frac)
    return trial.suggest_float(name, _clip(center - span, lo, hi), _clip(center + span, lo, hi))


def suggest_shared_hmm_params(trial: optuna.Trial, *, thorough: bool = False) -> dict:
    if thorough:
        return dict(
            n_init=trial.suggest_int("n_init", 12, 64),
            T_mcmc=trial.suggest_float("T_mcmc", 5e-4, 2.0, log=True),
            T_min=trial.suggest_float("T_min", 1e-5, 0.05, log=True),
            sigma_fraction=trial.suggest_float("sigma_fraction", 5e-4, 0.15, log=True),
            wide_sigma_fraction=trial.suggest_float("wide_sigma_fraction", 0.08, 0.85),
            temperature=trial.suggest_float("temperature", 0.05, 1.2),
            hmm_window=trial.suggest_int("hmm_window", 2, 16),
            clone_noise=trial.suggest_float("clone_noise", 0.005, 0.25),
            p_cat_step=trial.suggest_float("p_cat_step", 0.0, 0.6),
            kde_tau=trial.suggest_float("kde_tau", 0.005, 0.25),
            burnin_fraction=trial.suggest_float("burnin_fraction", 0.0, 0.35),
            anneal_T=trial.suggest_categorical("anneal_T", [True, False]),
            rejection_streak=trial.suggest_int("rejection_streak", 5, 30),
            hmm_lambda_noise=trial.suggest_float("hmm_lambda_noise", 1e-4, 0.05, log=True),
        )
    return dict(
        n_init=trial.suggest_int("n_init", 8, 48),
        T_mcmc=trial.suggest_float("T_mcmc", 1e-3, 1.0, log=True),
        sigma_fraction=trial.suggest_float("sigma_fraction", 1e-3, 0.1, log=True),
        wide_sigma_fraction=trial.suggest_float("wide_sigma_fraction", 0.1, 0.7),
        temperature=trial.suggest_float("temperature", 0.1, 1.0),
        hmm_window=trial.suggest_int("hmm_window", 2, 12),
        clone_noise=trial.suggest_float("clone_noise", 0.01, 0.2),
        p_cat_step=trial.suggest_float("p_cat_step", 0.0, 0.5),
        kde_tau=trial.suggest_float("kde_tau", 0.01, 0.2),
        burnin_fraction=trial.suggest_float("burnin_fraction", 0.0, 0.3),
        anneal_T=trial.suggest_categorical("anneal_T", [True, False]),
    )


def suggest_shared_hmm_params_refine(trial: optuna.Trial, center: dict) -> dict:
    return dict(
        n_init=_narrow_int(trial, "n_init", int(center["n_init"]), 12, 64),
        T_mcmc=_narrow_float(trial, "T_mcmc", float(center["T_mcmc"]), 5e-4, 2.0, log=True),
        T_min=_narrow_float(
            trial,
            "T_min",
            float(center.get("T_min", center["T_mcmc"] * 0.01)),
            1e-5,
            0.05,
            log=True,
        ),
        sigma_fraction=_narrow_float(
            trial, "sigma_fraction", float(center["sigma_fraction"]), 5e-4, 0.15, log=True
        ),
        wide_sigma_fraction=_narrow_float(
            trial, "wide_sigma_fraction", float(center["wide_sigma_fraction"]), 0.08, 0.85
        ),
        temperature=_narrow_float(trial, "temperature", float(center["temperature"]), 0.05, 1.2),
        hmm_window=_narrow_int(trial, "hmm_window", int(center["hmm_window"]), 2, 16),
        clone_noise=_narrow_float(trial, "clone_noise", float(center["clone_noise"]), 0.005, 0.25),
        p_cat_step=_narrow_float(trial, "p_cat_step", float(center["p_cat_step"]), 0.0, 0.6),
        kde_tau=_narrow_float(trial, "kde_tau", float(center["kde_tau"]), 0.005, 0.25, log=True),
        burnin_fraction=_narrow_float(
            trial, "burnin_fraction", float(center["burnin_fraction"]), 0.0, 0.35
        ),
        anneal_T=trial.suggest_categorical("anneal_T", [True, False]),
        rejection_streak=_narrow_int(
            trial, "rejection_streak", int(center.get("rejection_streak", 10)), 5, 30
        ),
        hmm_lambda_noise=_narrow_float(
            trial,
            "hmm_lambda_noise",
            float(center.get("hmm_lambda_noise", 0.01)),
            1e-4,
            0.05,
            log=True,
        ),
    )


def suggest_hmm_mcmc(trial: optuna.Trial) -> dict:
    return {**HMM_FIXED, **suggest_shared_hmm_params(trial)}



def suggest_hmm_mcmc_bw(trial: optuna.Trial) -> dict:
    shared = suggest_shared_hmm_params(trial)
    bw_only = dict(
        bw_prior_strength=trial.suggest_float("bw_prior_strength", 1.0, 50.0),
        bw_refit_every=trial.suggest_int("bw_refit_every", 3, 10),
        bw_min_obs=trial.suggest_int("bw_min_obs", 8, 24),
        bw_exploit_prior_scale=trial.suggest_float("bw_exploit_prior_scale", 1.0, 5.0),
    )
    return {**HMM_BW_FIXED, **shared, **bw_only}


def suggest_hmm_mcmc_test(trial: optuna.Trial) -> dict:
    shared = suggest_shared_hmm_params(trial)
    test_only = dict(
        bw_prior_strength=trial.suggest_float("bw_prior_strength", 1.0, 50.0),
        bw_refit_every=trial.suggest_int("bw_refit_every", 3, 10),
        bw_min_obs=trial.suggest_int("bw_min_obs", 8, 24),
        bw_exploit_prior_scale=trial.suggest_float("bw_exploit_prior_scale", 1.0, 5.0),
        spline_knots=trial.suggest_int("spline_knots", 10, 80),
        spline_floor=trial.suggest_float("spline_floor", 0.01, 0.2),
        spline_min_archive=trial.suggest_int("spline_min_archive", 5, 80),
        locality_sigma_fraction=trial.suggest_float("locality_sigma_fraction", 0.02, 0.2),
        spline_mix_scale=trial.suggest_float("spline_mix_scale", 0.5, 2.0),
    )
    return {**HMM_TEST_FIXED, **shared, **test_only}



def suggest_shared_hmm_params_ram(trial: optuna.Trial) -> dict:
    """Shared HMM knobs for RAM meta-tuning (overfit-prone params live in HMM_RAM_FIXED)."""
    return dict(
        T_mcmc=trial.suggest_float("T_mcmc", 5e-4, 2.0, log=True),
        T_min=trial.suggest_float("T_min", 1e-5, 0.05, log=True),
        sigma_fraction=trial.suggest_float("sigma_fraction", 5e-4, 0.15, log=True),
        temperature=trial.suggest_float("temperature", 0.05, 1.2),
        hmm_window=trial.suggest_int("hmm_window", 2, 16),
        clone_noise=trial.suggest_float("clone_noise", 0.005, 0.25),
        p_cat_step=trial.suggest_float("p_cat_step", 0.0, 0.6),
        kde_tau=trial.suggest_float("kde_tau", 0.005, 0.25),
        anneal_T=trial.suggest_categorical("anneal_T", [True, False]),
        rejection_streak=trial.suggest_int("rejection_streak", 5, 30),
        hmm_lambda_noise=trial.suggest_float("hmm_lambda_noise", 1e-4, 0.05, log=True),
    )


def suggest_hmm_mcmc_ram(trial: optuna.Trial) -> dict:
    shared = suggest_shared_hmm_params_ram(trial)
    ram_only = dict(
        ram_gamma=trial.suggest_float("ram_gamma", 0.55, 0.85),
        ram_s_min=trial.suggest_float("ram_s_min", 0.001, 0.02, log=True),
        ram_s_max=trial.suggest_float("ram_s_max", 0.2, 0.8),
    )
    return {**HMM_RAM_FIXED, **shared, **ram_only}


def suggest_shared_hmm_params_gp_mala(trial: optuna.Trial) -> dict:
    """Shared HMM knobs for GP-MALA meta-tuning (overfit-prone params live in HMM_GP_MALA_FIXED)."""
    return dict(
        T_mcmc=trial.suggest_float("T_mcmc", 5e-4, 2.0, log=True),
        T_min=trial.suggest_float("T_min", 1e-5, 0.05, log=True),
        sigma_fraction=trial.suggest_float("sigma_fraction", 5e-4, 0.15, log=True),
        temperature=trial.suggest_float("temperature", 0.05, 1.2),
        hmm_window=trial.suggest_int("hmm_window", 2, 16),
        clone_noise=trial.suggest_float("clone_noise", 0.005, 0.25),
        p_cat_step=trial.suggest_float("p_cat_step", 0.0, 0.6),
        kde_tau=trial.suggest_float("kde_tau", 0.005, 0.25),
        anneal_T=trial.suggest_categorical("anneal_T", [True, False]),
        rejection_streak=trial.suggest_int("rejection_streak", 5, 30),
        hmm_lambda_noise=trial.suggest_float("hmm_lambda_noise", 1e-4, 0.05, log=True),
    )


def suggest_hmm_mcmc_gp_mala(trial: optuna.Trial) -> dict:
    shared = suggest_shared_hmm_params_gp_mala(trial)
    gp_mala_only = dict(
        mala_step_size=trial.suggest_float("mala_step_size", 0.02, 0.3, log=True),
        gp_kappa_exploit=trial.suggest_float("gp_kappa_exploit", 0.1, 3.0),
        gp_kappa_explore=trial.suggest_float("gp_kappa_explore", 1.0, 8.0),
        gp_target_temperature=trial.suggest_float("gp_target_temperature", 0.3, 2.0),
        ram_gamma=trial.suggest_float("ram_gamma", 0.55, 0.85),
        ram_s_min=trial.suggest_float("ram_s_min", 0.01, 0.08),
        ram_s_max=trial.suggest_float("ram_s_max", 0.2, 0.8),
        tr_length_init=trial.suggest_float("tr_length_init", 0.4, 1.6),
        tr_failure_tol=trial.suggest_int("tr_failure_tol", 2, 8),
        tr_success_tol=trial.suggest_int("tr_success_tol", 2, 5),
        acq_greedy_after=trial.suggest_float("acq_greedy_after", 0.7, 0.95),
        tr_restart_kappa_start=trial.suggest_float("tr_restart_kappa_start", 1.0, 5.0),
        bo_warmup_frac=trial.suggest_float("bo_warmup_frac", 0.0, 0.4),
    )
    return {**HMM_GP_MALA_FIXED, **shared, **gp_mala_only}


def suggest_hmm_mcmc_hmc(trial: optuna.Trial) -> dict:
    hmc_only = dict(
        rejection_streak=trial.suggest_int("rejection_streak", 5, 20),
        hmm_window=trial.suggest_int("hmm_window", 4, 16),
        hmm_refit_every=trial.suggest_int("hmm_refit_every", 2, 10),
        kappa_exploit=trial.suggest_float("kappa_exploit", 0.1, 2.0),
        kappa_explore=trial.suggest_float("kappa_explore", 1.0, 6.0),
        kappa_escape=trial.suggest_float("kappa_escape", 2.0, 8.0),
        T_exploit=trial.suggest_float("T_exploit", 0.1, 1.0, log=True),
        T_explore=trial.suggest_float("T_explore", 0.5, 2.0),
        T_escape=trial.suggest_float("T_escape", 1.0, 4.0),
        step_size_init=trial.suggest_float("step_size_init", 0.02, 0.3, log=True),
        target_accept=trial.suggest_float("target_accept", 0.65, 0.9),
        nuts_max_tree_depth=trial.suggest_int("nuts_max_tree_depth", 6, 12),
        bo_warmup_frac=trial.suggest_float("bo_warmup_frac", 0.0, 0.4),
        burst_steps_escape=trial.suggest_int("burst_steps_escape", 0, 96, step=16),
        burst_steps_stuck=trial.suggest_int("burst_steps_stuck", 0, 64, step=16),
        hmm_switch_confirm=trial.suggest_int("hmm_switch_confirm", 1, 4),
        gp_refit_every=trial.suggest_int("gp_refit_every", 1, 4),
        gp_noise_floor=trial.suggest_float("gp_noise_floor", 1e-8, 1e-4, log=True),
        explore_beta=trial.suggest_float("explore_beta", 1.0, 8.0),
        gp_subsample=trial.suggest_categorical("gp_subsample", [False, True]),
        exploit_argmax_every=trial.suggest_int("exploit_argmax_every", 0, 5),
        gp_argmax_raw_samples=trial.suggest_int("gp_argmax_raw_samples", 32, 128),
        acq_greedy_after=trial.suggest_float("acq_greedy_after", 0.7, 0.95),
        p_cat_step=trial.suggest_float("p_cat_step", 0.0, 0.4),
    )
    return {**HMM_HMC_FIXED, **hmc_only}


def suggest_hmm_mcmc_dream(trial: optuna.Trial) -> dict:
    shared = suggest_shared_hmm_params(trial, thorough=THOROUGH_MODE)
    dream_only = dict(
        n_chains=trial.suggest_int("n_chains", 2, 8),
        orchestrate_every=trial.suggest_int("orchestrate_every", 2, 25),
        orchestrate_patience=trial.suggest_int("orchestrate_patience", 8, 40),
        p_dream=trial.suggest_float("p_dream", 0.1, 0.9),
        dream_n_pairs=trial.suggest_int("dream_n_pairs", 1, 4),
        dream_cr=trial.suggest_float("dream_cr", 0.3, 1.0),
        dream_gamma1_prob=trial.suggest_float("dream_gamma1_prob", 0.0, 0.35),
        dream_eps=trial.suggest_float("dream_eps", 5e-5, 0.02, log=True),
        dream_min_pop=trial.suggest_int("dream_min_pop", 2, 12),
        dream_diversity_frac=trial.suggest_float("dream_diversity_frac", 0.0, 0.6),
        bw_prior_strength=trial.suggest_float("bw_prior_strength", 0.5, 60.0),
        bw_refit_every=trial.suggest_int("bw_refit_every", 2, 12),
        bw_min_obs=trial.suggest_int("bw_min_obs", 6, 32),
        bw_exploit_prior_scale=trial.suggest_float("bw_exploit_prior_scale", 0.5, 6.0),
        bw_n_em_iters=trial.suggest_int("bw_n_em_iters", 1, 8),
        bw_max_len=trial.suggest_int("bw_max_len", 32, 128),
    )
    return {**HMM_DREAM_FIXED, **shared, **dream_only}


def suggest_hmm_mcmc_dream_refine(trial: optuna.Trial, center: dict) -> dict:
    shared = suggest_shared_hmm_params_refine(trial, center)
    dream_only = dict(
        n_chains=_narrow_int(trial, "n_chains", int(center["n_chains"]), 2, 8),
        orchestrate_every=_narrow_int(
            trial, "orchestrate_every", int(center["orchestrate_every"]), 2, 25
        ),
        orchestrate_patience=_narrow_int(
            trial, "orchestrate_patience", int(center.get("orchestrate_patience", 15)), 8, 40
        ),
        p_dream=_narrow_float(trial, "p_dream", float(center["p_dream"]), 0.1, 0.9),
        dream_n_pairs=_narrow_int(trial, "dream_n_pairs", int(center["dream_n_pairs"]), 1, 4),
        dream_cr=_narrow_float(trial, "dream_cr", float(center["dream_cr"]), 0.3, 1.0),
        dream_gamma1_prob=_narrow_float(
            trial, "dream_gamma1_prob", float(center["dream_gamma1_prob"]), 0.0, 0.35
        ),
        dream_eps=_narrow_float(trial, "dream_eps", float(center["dream_eps"]), 5e-5, 0.02, log=True),
        dream_min_pop=_narrow_int(trial, "dream_min_pop", int(center["dream_min_pop"]), 2, 12),
        dream_diversity_frac=_narrow_float(
            trial,
            "dream_diversity_frac",
            float(center.get("dream_diversity_frac", 0.25)),
            0.0,
            0.6,
        ),
        bw_prior_strength=_narrow_float(
            trial, "bw_prior_strength", float(center["bw_prior_strength"]), 0.5, 60.0
        ),
        bw_refit_every=_narrow_int(trial, "bw_refit_every", int(center["bw_refit_every"]), 2, 12),
        bw_min_obs=_narrow_int(trial, "bw_min_obs", int(center["bw_min_obs"]), 6, 32),
        bw_exploit_prior_scale=_narrow_float(
            trial, "bw_exploit_prior_scale", float(center["bw_exploit_prior_scale"]), 0.5, 6.0
        ),
        bw_n_em_iters=_narrow_int(trial, "bw_n_em_iters", int(center.get("bw_n_em_iters", 3)), 1, 8),
        bw_max_len=_narrow_int(trial, "bw_max_len", int(center.get("bw_max_len", 64)), 32, 128),
    )
    return {**HMM_DREAM_FIXED, **shared, **dream_only}


def params_from_study(study: optuna.Study, fixed: dict) -> dict:
    return {**fixed, **study.best_trial.params}


In [7]:
# =============================================================================
# Optuna meta-studies
# =============================================================================


def _trial_params_only(params: dict, fixed: dict) -> dict:
    """Keep only keys that Optuna should suggest (exclude fixed flags)."""
    fixed_keys = set(fixed.keys())
    return {k: v for k, v in params.items() if k not in fixed_keys}


def _dream_warm_start_params(loaded: dict, fixed: dict) -> dict:
    params = _trial_params_only(loaded, fixed)
    params.setdefault("T_min", float(loaded.get("T_min", loaded["T_mcmc"] * 0.01)))
    params.setdefault("rejection_streak", int(loaded.get("rejection_streak", 10)))
    params.setdefault("hmm_lambda_noise", float(loaded.get("hmm_lambda_noise", 0.01)))
    params.setdefault("bw_n_em_iters", int(loaded.get("bw_n_em_iters", 3)))
    params.setdefault("bw_max_len", int(loaded.get("bw_max_len", 64)))
    return params


def _load_warm_start(algo_name: str, fixed: dict) -> dict | None:
    if not WARM_START_FROM_BEST or algo_name != "HMM_MCMC_DREAM":
        return None
    path = RESULTS_DIR / "best_params_hmm_mcmc_dream.json"
    if not path.is_file():
        return None
    with path.open(encoding="utf-8") as fh:
        loaded = json.load(fh)
    print(f"Warm-start enqueue from {path.name}")
    return _dream_warm_start_params(loaded, fixed)


def run_meta_study(
    algo_name: str,
    algo_cls: Type,
    suggest_fn,
    *,
    fixed: dict,
    n_trials: int | None = None,
    warm_start: dict | None = None,
    study_suffix: str = "",
) -> optuna.Study:
    n_trials = n_trials or N_META_TRIALS

    def objective(trial: optuna.Trial) -> float:
        params = suggest_fn(trial)
        return meta_objective(
            algo_cls,
            params,
            TUNE_SEEDS,
            desc=f"{algo_name} trial={trial.number}",
        )

    study = optuna.create_study(
        direction="maximize",
        study_name=f"tune_{algo_name}_lcbench{study_suffix}",
        sampler=TPESampler(
            seed=META_SEED,
            n_startup_trials=min(META_STARTUP_TRIALS, n_trials),
            multivariate=True,
        ),
    )

    if warm_start is not None:
        study.enqueue_trial(_trial_params_only(warm_start, fixed))

    for _ in tqdm(range(n_trials), desc=f"Meta-study {algo_name}{study_suffix}", disable=not USE_TQDM):
        study.optimize(objective, n_trials=1, show_progress_bar=False)

    print(f"\n{algo_name}{study_suffix}: best val_accuracy = {study.best_value:.6f}")
    print("Best params:")
    for k, v in study.best_trial.params.items():
        print(f"  {k}: {v}")
    return study


def run_dream_refine_study(coarse_best: dict) -> optuna.Study | None:
    if N_REFINE_TRIALS <= 0:
        return None

    def suggest_refine(trial: optuna.Trial) -> dict:
        return suggest_hmm_mcmc_dream_refine(trial, coarse_best)

    study = run_meta_study(
        "HMM_MCMC_DREAM",
        HMM_MCMC_DREAM,
        suggest_refine,
        fixed=HMM_DREAM_FIXED,
        n_trials=N_REFINE_TRIALS,
        warm_start=coarse_best,
        study_suffix="_refine",
    )
    return study


_META_REGISTRY = {
    "HMM_MCMC": (HMM_MCMC, suggest_hmm_mcmc, HMM_FIXED),
    "HMM_MCMC_BW": (HMM_MCMC_BW, suggest_hmm_mcmc_bw, HMM_BW_FIXED),
    "HMM_MCMC_RAM": (HMM_MCMC_RAM, suggest_hmm_mcmc_ram, HMM_RAM_FIXED),
    "HMM_MCMC_GP_MALA": (HMM_MCMC_GP_MALA, suggest_hmm_mcmc_gp_mala, HMM_GP_MALA_FIXED),
    "HMM_MCMC_HMC": (HMM_MCMC_HMC, suggest_hmm_mcmc_hmc, HMM_HMC_FIXED),
    "HMM_MCMC_TEST": (HMM_MCMC_TEST, suggest_hmm_mcmc_test, HMM_TEST_FIXED),
    "HMM_MCMC_DREAM": (HMM_MCMC_DREAM, suggest_hmm_mcmc_dream, HMM_DREAM_FIXED),
}

studies: dict[str, optuna.Study] = {}
best_params: dict[str, dict] = {}

for algo_name in TUNE_ALGORITHMS:
    algo_cls, suggest_fn, fixed = _META_REGISTRY[algo_name]
    warm = _load_warm_start(algo_name, fixed)
    studies[algo_name] = run_meta_study(
        algo_name,
        algo_cls,
        suggest_fn,
        fixed=fixed,
        warm_start=warm,
    )
    best_params[algo_name] = params_from_study(studies[algo_name], fixed)

    if algo_name == "HMM_MCMC_DREAM" and N_REFINE_TRIALS > 0:
        coarse_best = dict(best_params[algo_name])
        coarse_score = studies[algo_name].best_value
        study_refine = run_dream_refine_study(coarse_best)
        if study_refine is not None and study_refine.best_value >= coarse_score:
            studies["HMM_MCMC_DREAM_refine"] = study_refine
            best_params[algo_name] = params_from_study(study_refine, fixed)
            print(
                f"Refine improved meta objective: {study_refine.best_value:.6f} "
                f"(was {coarse_score:.6f})"
            )
        elif study_refine is not None:
            print(
                f"Refine did not beat coarse best: {study_refine.best_value:.6f} "
                f"vs {coarse_score:.6f}; keeping coarse params"
            )

study_hmm = studies.get("HMM_MCMC")
study_bw = studies.get("HMM_MCMC_BW")
study_ram = studies.get("HMM_MCMC_RAM")
study_gp_mala = studies.get("HMM_MCMC_GP_MALA")
study_hmc = studies.get("HMM_MCMC_HMC")
study_test = studies.get("HMM_MCMC_TEST")
study_dream = studies.get("HMM_MCMC_DREAM_refine") or studies.get("HMM_MCMC_DREAM")
study_dream_refine = studies.get("HMM_MCMC_DREAM_refine")
study_dream_coarse = studies.get("HMM_MCMC_DREAM")

best_params_hmm = best_params.get("HMM_MCMC")
best_params_bw = best_params.get("HMM_MCMC_BW")
best_params_ram = best_params.get("HMM_MCMC_RAM")
best_params_gp_mala = best_params.get("HMM_MCMC_GP_MALA")
best_params_hmc = best_params.get("HMM_MCMC_HMC")
best_params_test = best_params.get("HMM_MCMC_TEST")
best_params_dream = best_params.get("HMM_MCMC_DREAM")

# Load previously tuned params when skipping re-tuning
if best_params_hmm is None and (RESULTS_DIR / "best_params_hmm_mcmc.json").is_file():
    with (RESULTS_DIR / "best_params_hmm_mcmc.json").open(encoding="utf-8") as fh:
        best_params_hmm = json.load(fh)
        print("Loaded existing best_params_hmm_mcmc.json")
if best_params_bw is None and (RESULTS_DIR / "best_params_hmm_mcmc_bw.json").is_file():
    with (RESULTS_DIR / "best_params_hmm_mcmc_bw.json").open(encoding="utf-8") as fh:
        best_params_bw = json.load(fh)
        print("Loaded existing best_params_hmm_mcmc_bw.json")
if best_params_ram is None and (RESULTS_DIR / "best_params_hmm_mcmc_ram.json").is_file():
    with (RESULTS_DIR / "best_params_hmm_mcmc_ram.json").open(encoding="utf-8") as fh:
        best_params_ram = json.load(fh)
        print("Loaded existing best_params_hmm_mcmc_ram.json")
if best_params_gp_mala is None and (RESULTS_DIR / "best_params_hmm_mcmc_gp_mala.json").is_file():
    with (RESULTS_DIR / "best_params_hmm_mcmc_gp_mala.json").open(encoding="utf-8") as fh:
        best_params_gp_mala = json.load(fh)
        print("Loaded existing best_params_hmm_mcmc_gp_mala.json")
if best_params_hmc is None and (RESULTS_DIR / "best_params_hmm_mcmc_hmc.json").is_file():
    with (RESULTS_DIR / "best_params_hmm_mcmc_hmc.json").open(encoding="utf-8") as fh:
        best_params_hmc = json.load(fh)
        print("Loaded existing best_params_hmm_mcmc_hmc.json")
if best_params_test is None and (RESULTS_DIR / "best_params_hmm_mcmc_test.json").is_file():
    with (RESULTS_DIR / "best_params_hmm_mcmc_test.json").open(encoding="utf-8") as fh:
        best_params_test = json.load(fh)
        print("Loaded existing best_params_hmm_mcmc_test.json")


c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\.venv\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(


Meta-study HMM_MCMC_GP_MALA:   0%|          | 0/120 [00:00<?, ?it/s]

HMM_MCMC_GP_MALA trial=0:   0%|          | 0/9 [00:00<?, ?it/s]

[W 2026-07-09 15:48:12,920] Trial 0 failed with parameters: {'T_mcmc': 0.011170843861578643, 'T_min': 0.03285970816964246, 'sigma_fraction': 0.03252428484490637, 'temperature': 0.7384572568265921, 'hmm_window': 4, 'clone_noise': 0.043218657482369645, 'p_cat_step': 0.03485016730091967, 'kde_tau': 0.2172131557148591, 'anneal_T': False, 'rejection_streak': 5, 'hmm_lambda_noise': 0.04147225000481637, 'mala_step_size': 0.10779361932748845, 'gp_kappa_exploit': 0.7157834209670008, 'gp_kappa_explore': 2.272774770449704, 'gp_refit_every': 4, 'ram_gamma': 0.6412726728878614, 'ram_s_min': 0.004816414530907085, 'ram_s_max': 0.4591670111852695} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_36

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# Validation on held-out seeds
# =============================================================================


def validate_params(algo_name: str, algo_cls: Type, params: dict) -> dict:
    scores: list[float] = []
    runs = [(inst, seed) for inst in LCBENCH_INSTANCES for seed in VAL_SEEDS]
    pbar = tqdm(runs, desc=f"Validate {algo_name}", disable=not USE_TQDM)
    for inst, seed in pbar:
        acc = run_hmm_once(algo_cls, contexts[inst], params, N_EVALS, seed)
        scores.append(acc)
        pbar.set_postfix(inst=inst, seed=seed, acc=f"{acc:.4f}")
    arr = np.array(scores, dtype=float)
    return {
        "algorithm": algo_name,
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr)),
        "best": float(np.max(arr)),
        "worst": float(np.min(arr)),
        "scores": scores,
    }


validation_rows = []

if best_params_hmm is not None:
    if study_hmm is not None:
        validation_rows.append(
            validate_params("HMM_MCMC (tuned)", HMM_MCMC, best_params_hmm)
        )
    validation_rows.append(
        validate_params("HMM_MCMC (default)", HMM_MCMC, HMM_DEFAULT)
    )


if best_params_bw is not None:
    if study_bw is not None:
        validation_rows.append(
            validate_params("HMM_MCMC_BW (tuned)", HMM_MCMC_BW, best_params_bw)
        )
    validation_rows.append(
        validate_params("HMM_MCMC_BW (default)", HMM_MCMC_BW, HMM_BW_DEFAULT)
    )

if best_params_test is not None:
    if study_test is not None:
        validation_rows.append(
            validate_params("HMM_MCMC_TEST (tuned)", HMM_MCMC_TEST, best_params_test)
        )
    validation_rows.append(
        validate_params("HMM_MCMC_TEST (default)", HMM_MCMC_TEST, HMM_TEST_DEFAULT)
    )

if best_params_ram is not None:
    if study_ram is not None:
        validation_rows.append(
            validate_params("HMM_MCMC_RAM (tuned)", HMM_MCMC_RAM, best_params_ram)
        )
    validation_rows.append(
        validate_params("HMM_MCMC_RAM (default)", HMM_MCMC_RAM, HMM_RAM_DEFAULT)
    )

if best_params_gp_mala is not None:
    if study_gp_mala is not None:
        validation_rows.append(
            validate_params("HMM_MCMC_GP_MALA (tuned)", HMM_MCMC_GP_MALA, best_params_gp_mala)
        )
    validation_rows.append(
        validate_params("HMM_MCMC_GP_MALA (default)", HMM_MCMC_GP_MALA, HMM_GP_MALA_DEFAULT)
    )

if best_params_hmc is not None:
    if study_hmc is not None:
        validation_rows.append(
            validate_params("HMM_MCMC_HMC (tuned)", HMM_MCMC_HMC, best_params_hmc)
        )
    validation_rows.append(
        validate_params("HMM_MCMC_HMC (default)", HMM_MCMC_HMC, HMM_HMC_DEFAULT)
    )

if best_params_dream is not None:
    if study_dream is not None:
        validation_rows.append(
            validate_params("HMM_MCMC_DREAM (tuned)", HMM_MCMC_DREAM, best_params_dream)
        )
    validation_rows.append(
        validate_params("HMM_MCMC_DREAM (default)", HMM_MCMC_DREAM, HMM_DREAM_DEFAULT)
    )
validation_df = pd.DataFrame([
    {k: v for k, v in row.items() if k != "scores"} for row in validation_rows
])
print(validation_df.to_string(index=False))


Validate HMM_MCMC (default):   0%|          | 0/12 [00:00<?, ?it/s]

H-MCMC-FMP:   0%|          | 0/200 [00:00<?, ?it/s]


HMM MCMC State History (last run):
     Eval  Chain    State       Loss  Accepted  Rej_Streak
0      33      0  EXPLOIT -98.080330     False           1
1      34      0  EXPLOIT -99.330772      True           0
2      35      0  EXPLOIT -99.331528      True           0
3      36      0  EXPLOIT -99.328117      True           0
4      37      0  EXPLOIT -98.078835     False           1
5      38      0  EXPLOIT -92.511612     False           2
6      39      0  EXPLORE -99.316757     False           3
7      40      0  EXPLOIT -99.324707      True           0
8      41      0  EXPLOIT -98.059006     False           1
9      42      0  EXPLOIT -98.047409     False           2
10     43      0  EXPLOIT -99.324707      True           0
11     44      0  EXPLOIT -99.312965      True           0
12     45      0  EXPLOIT -98.034691     False           1
13     46      0  EXPLOIT -99.312965      True           0
14     47      0  EXPLOIT -98.059006     False           1
15     48      0  EX

H-MCMC-FMP:   0%|          | 0/200 [00:00<?, ?it/s]


HMM MCMC State History (last run):
     Eval  Chain    State       Loss  Accepted  Rej_Streak
0      33      0  EXPLOIT -96.637115     False           1
1      34      0  EXPLOIT -99.330772      True           0
2      35      0  EXPLOIT -98.145073     False           1
3      36      0  EXPLOIT -99.330772      True           0
4      37      0  EXPLOIT -98.028709     False           1
5      38      0  EXPLOIT -99.330772      True           0
6      39      0  EXPLOIT -99.330772      True           0
7      40      0  EXPLOIT -99.360710      True           0
8      41      0  EXPLOIT -99.366020      True           0
9      42      0  EXPLOIT -99.366020      True           0
10     43      0  EXPLOIT -99.366020      True           0
11     44      0  EXPLOIT -99.364502      True           0
12     45      0  EXPLOIT -99.368294      True           0
13     46      0  EXPLOIT -97.337112     False           1
14     47      0  EXPLOIT -98.237595     False           2
15     48      0  EX

...

Validate HMM_MCMC_BW (default):   0%|          | 0/12 [00:00<?, ?it/s]

Validate HMM_MCMC_TEST (default):   0%|          | 0/12 [00:00<?, ?it/s]

Validate HMM_MCMC_RAM (tuned):   0%|          | 0/12 [00:00<?, ?it/s]

Validate HMM_MCMC_RAM (default):   0%|          | 0/12 [00:00<?, ?it/s]

              algorithm      mean      std      best     worst
     HMM_MCMC (default) 92.863836 9.344148 99.547371 77.255966
  HMM_MCMC_BW (default) 92.647641 9.535955 99.501053 78.726379
HMM_MCMC_TEST (default) 94.226263 7.518163 99.454758 79.234497
   HMM_MCMC_RAM (tuned) 94.363814 6.728558 99.624489 82.849091
 HMM_MCMC_RAM (default) 94.773738 6.760667 99.710037 83.693733


In [ ]:
# =============================================================================
# Save results + Optuna plots
# =============================================================================

# JSON best params (only overwrite algorithms tuned in this run)
if best_params_hmm is not None:
    hmm_json = RESULTS_DIR / "best_params_hmm_mcmc.json"
    with hmm_json.open("w", encoding="utf-8") as fh:
        json.dump(best_params_hmm, fh, indent=2, default=str)
    print(f"Saved {hmm_json}")
if best_params_bw is not None:
    bw_json = RESULTS_DIR / "best_params_hmm_mcmc_bw.json"
    with bw_json.open("w", encoding="utf-8") as fh:
        json.dump(best_params_bw, fh, indent=2, default=str)
    print(f"Saved {bw_json}")
if best_params_test is not None:
    test_json = RESULTS_DIR / "best_params_hmm_mcmc_test.json"
    with test_json.open("w", encoding="utf-8") as fh:
        json.dump(best_params_test, fh, indent=2, default=str)
    print(f"Saved {test_json}")
if best_params_ram is not None:
    ram_json = RESULTS_DIR / "best_params_hmm_mcmc_ram.json"
    with ram_json.open("w", encoding="utf-8") as fh:
        json.dump(best_params_ram, fh, indent=2, default=str)
    print(f"Saved {ram_json}")
if best_params_gp_mala is not None:
    gp_mala_json = RESULTS_DIR / "best_params_hmm_mcmc_gp_mala.json"
    with gp_mala_json.open("w", encoding="utf-8") as fh:
        json.dump(best_params_gp_mala, fh, indent=2, default=str)
    print(f"Saved {gp_mala_json}")
if best_params_hmc is not None:
    hmc_json = RESULTS_DIR / "best_params_hmm_mcmc_hmc.json"
    with hmc_json.open("w", encoding="utf-8") as fh:
        json.dump(best_params_hmc, fh, indent=2, default=str)
    print(f"Saved {hmc_json}")
if best_params_dream is not None:
    dream_json = RESULTS_DIR / "best_params_hmm_mcmc_dream.json"
    with dream_json.open("w", encoding="utf-8") as fh:
        json.dump(best_params_dream, fh, indent=2, default=str)
    print(f"Saved {dream_json}")

# Summary table
summary_rows = []
if study_hmm is not None and best_params_hmm is not None:
    tuned_hmm = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC (tuned)")
    default_hmm = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC (default)")
    summary_rows.extend([
        {
            "algorithm": "HMM_MCMC",
            "variant": "tuned",
            "meta_best": study_hmm.best_value,
            "val_mean": tuned_hmm["mean"],
            "val_std": tuned_hmm["std"],
            "val_best": tuned_hmm["best"],
        },
        {
            "algorithm": "HMM_MCMC",
            "variant": "default",
            "meta_best": float("nan"),
            "val_mean": default_hmm["mean"],
            "val_std": default_hmm["std"],
            "val_best": default_hmm["best"],
        },
    ])

if study_bw is not None and best_params_bw is not None:
    tuned_bw = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_BW (tuned)")
    default_bw = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_BW (default)")
    summary_rows.extend([
        {
            "algorithm": "HMM_MCMC_BW",
            "variant": "tuned",
            "meta_best": study_bw.best_value,
            "val_mean": tuned_bw["mean"],
            "val_std": tuned_bw["std"],
            "val_best": tuned_bw["best"],
        },
        {
            "algorithm": "HMM_MCMC_BW",
            "variant": "default",
            "meta_best": float("nan"),
            "val_mean": default_bw["mean"],
            "val_std": default_bw["std"],
            "val_best": default_bw["best"],
        },
    ])
if study_test is not None and best_params_test is not None:
    tuned_test = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_TEST (tuned)")
    default_test = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_TEST (default)")
    summary_rows.extend([
        {
            "algorithm": "HMM_MCMC_TEST",
            "variant": "tuned",
            "meta_best": study_test.best_value,
            "val_mean": tuned_test["mean"],
            "val_std": tuned_test["std"],
            "val_best": tuned_test["best"],
        },
        {
            "algorithm": "HMM_MCMC_TEST",
            "variant": "default",
            "meta_best": float("nan"),
            "val_mean": default_test["mean"],
            "val_std": default_test["std"], 
            
            "val_best": default_test["best"],
        },
    ])
if study_ram is not None and best_params_ram is not None:
    tuned_ram = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_RAM (tuned)")
    default_ram = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_RAM (default)")
    summary_rows.extend([
        {
            "algorithm": "HMM_MCMC_RAM",
            "variant": "tuned",
            "meta_best": study_ram.best_value,
            "val_mean": tuned_ram["mean"],
            "val_std": tuned_ram["std"],
            "val_best": tuned_ram["best"],
        },
        {
            "algorithm": "HMM_MCMC_RAM",
            "variant": "default",
            "meta_best": float("nan"),
            "val_mean": default_ram["mean"],
            "val_std": default_ram["std"],
            "val_best": default_ram["best"],
        },
    ])

if study_gp_mala is not None and best_params_gp_mala is not None:
    tuned_gp_mala = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_GP_MALA (tuned)")
    default_gp_mala = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_GP_MALA (default)")
    summary_rows.extend([
        {
            "algorithm": "HMM_MCMC_GP_MALA",
            "variant": "tuned",
            "meta_best": study_gp_mala.best_value,
            "val_mean": tuned_gp_mala["mean"],
            "val_std": tuned_gp_mala["std"],
            "val_best": tuned_gp_mala["best"],
        },
        {
            "algorithm": "HMM_MCMC_GP_MALA",
            "variant": "default",
            "meta_best": float("nan"),
            "val_mean": default_gp_mala["mean"],
            "val_std": default_gp_mala["std"],
            "val_best": default_gp_mala["best"],
        },
    ])

if study_hmc is not None and best_params_hmc is not None:
    tuned_hmc = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_HMC (tuned)")
    default_hmc = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_HMC (default)")
    summary_rows.extend([
        {
            "algorithm": "HMM_MCMC_HMC",
            "variant": "tuned",
            "meta_best": study_hmc.best_value,
            "val_mean": tuned_hmc["mean"],
            "val_std": tuned_hmc["std"],
            "val_best": tuned_hmc["best"],
        },
        {
            "algorithm": "HMM_MCMC_HMC",
            "variant": "default",
            "meta_best": float("nan"),
            "val_mean": default_hmc["mean"],
            "val_std": default_hmc["std"],
            "val_best": default_hmc["best"],
        },
    ])

if study_dream is not None and best_params_dream is not None:
    tuned_dream = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_DREAM (tuned)")
    default_dream = next(r for r in validation_rows if r["algorithm"] == "HMM_MCMC_DREAM (default)")
    summary_rows.extend([
        {
            "algorithm": "HMM_MCMC_DREAM",
            "variant": "tuned",
            "meta_best": study_dream.best_value,
            "val_mean": tuned_dream["mean"],
            "val_std": tuned_dream["std"],
            "val_best": tuned_dream["best"],
        },
        {
            "algorithm": "HMM_MCMC_DREAM",
            "variant": "default",
            "meta_best": float("nan"),
            "val_mean": default_dream["mean"],
            "val_std": default_dream["std"],
            "val_best": default_dream["best"],
        },
    ])
    if study_dream_coarse is not None and study_dream_refine is not None:
        summary_rows.append({
            "algorithm": "HMM_MCMC_DREAM",
            "variant": "coarse_only",
            "meta_best": study_dream_coarse.best_value,
            "val_mean": float("nan"),
            "val_std": float("nan"),
            "val_best": float("nan"),
        })
summary_df = pd.DataFrame(summary_rows)
csv_path = RESULTS_DIR / "tuning_summary.csv"
tex_path = RESULTS_DIR / "tuning_summary.tex"
summary_df.to_csv(csv_path, index=False, float_format="%.6f")
with tex_path.open("w", encoding="utf-8") as fh:
    fh.write(summary_df.round(4).to_latex(index=False, escape=True))
print(f"Saved {csv_path}")
print(f"Saved {tex_path}")

# Optuna plots (Optuna >=4 returns Axes from matplotlib helpers; no ax= kwarg)
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

_plot_studies = []
if study_hmm is not None:
    _plot_studies.append((study_hmm, "hmm_mcmc"))
if study_bw is not None:
    _plot_studies.append((study_bw, "hmm_mcmc_bw"))
if study_ram is not None:
    _plot_studies.append((study_ram, "hmm_mcmc_ram"))
if study_gp_mala is not None:
    _plot_studies.append((study_gp_mala, "hmm_mcmc_gp_mala"))
if study_hmc is not None:
    _plot_studies.append((study_hmc, "hmm_mcmc_hmc"))
if study_test is not None:
    _plot_studies.append((study_test, "hmm_mcmc_test"))
if study_dream_coarse is not None:
    _plot_studies.append((study_dream_coarse, "hmm_mcmc_dream_coarse"))
if study_dream_refine is not None:
    _plot_studies.append((study_dream_refine, "hmm_mcmc_dream_refine"))
elif study_dream is not None and study_dream_coarse is None:
    _plot_studies.append((study_dream, "hmm_mcmc_dream"))

for study, label in _plot_studies:
    if len(study.trials) < 2:
        continue
    ax = plot_optimization_history(study)
    ax.set_title(f"Meta-study history: {label}")
    fig = ax.figure
    fig.set_size_inches(8, 4)
    fig.tight_layout()
    hist_path = PLOTS_DIR / f"optuna_history_{label}.png"
    fig.savefig(hist_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {hist_path}")

    try:
        ax = plot_param_importances(study)
        ax.set_title(f"Param importances: {label}")
        fig = ax.figure
        fig.set_size_inches(8, 5)
        fig.tight_layout()
        imp_path = PLOTS_DIR / f"optuna_importance_{label}.png"
        fig.savefig(imp_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved {imp_path}")
    except Exception as exc:
        print(f"Skipping importance plot for {label}: {exc}")

print("\nDone.")


Saved c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\experiments\yahpo_tuning_results\best_params_hmm_mcmc.json
Saved c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\experiments\yahpo_tuning_results\best_params_hmm_mcmc_bw.json
Saved c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\experiments\yahpo_tuning_results\best_params_hmm_mcmc_test.json
Saved c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\experiments\yahpo_tuning_results\best_params_hmm_mcmc_ram.json
Saved c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\experiments\yahpo_tuning_results\tuning_summary.csv
Saved c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\experiments\yahpo_tuning_results\tuning_summary.tex


C:\Users\Administrator\AppData\Local\Temp\ipykernel_16388\4072081504.py:182: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_optimization_history(study)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_16388\4072081504.py:193: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_param_importances(study)


Saved c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\experiments\yahpo_tuning_plots\optuna_history_hmm_mcmc_ram.png
Saved c:\Users\Administrator\Documents\!prog\HPO_RL test\HPO_RL\experiments\yahpo_tuning_plots\optuna_importance_hmm_mcmc_ram.png

Done.


## Next steps

1. **Thorough mode** (`THOROUGH_MODE = True`): 120 coarse meta-trials, 200 evals/trial, 3 tune seeds × 3 lcbench instances, multivariate TPE.
2. Set `SMOKE_MODE = True` only for a quick pipeline check (20 trials, 1 instance).
3. Set `TUNE_ALGORITHMS = ["HMM_MCMC_GP_MALA"]` (default) or add other variants to re-tune.
4. Best params are written to `experiments/yahpo_tuning_results/best_params_hmm_mcmc_gp_mala.json`.
5. [yahpo_hmm_mcmc_test_vs_optuna.ipynb](yahpo_hmm_mcmc_test_vs_optuna.ipynb) loads that JSON automatically — re-run after tuning.
